# Notebook 05 — First QLoRA SFT Run

This notebook walks through a complete first supervised fine-tuning run with the practical 2026 default:

- **4-bit QLoRA adapters**
- **Unsloth + TRL + PEFT**
- **small local conversational data**
- **evaluation-aware training on Colab Pro**
- **open-source checkpoints and adapters only**

The goal is not to win a benchmark. The goal is to make the full workflow visible end to end.

## What you will build

By the end of the notebook you will have:

1. created a tiny local conversation dataset
2. rendered it with the model chat template
3. made a disciplined train/validation split
 4. inspected the PEFT adapter parameter footprint
 5. run a short QLoRA SFT job
 6. inspected training logs and validation loss
 7. compared base vs tuned completions on held-out prompts
 8. saved the adapter for later notebooks

## Runtime framing

 The target environment is Google Colab Pro with an open-source 4-bit instruct checkpoint.

 Unsloth handles the fast loading path, TRL supplies the supervised fine-tuning trainer, and PEFT defines the adapter layer we save at the end.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Add notebook helpers and artifact paths

We keep the helpers explicit so every moving part stays inspectable.

In [ ]:
from peft import PeftModel

random.seed(5)

ARTIFACT_DIR = Path(OUTPUT_DIR) / "notebook_05_first_qlora_sft_run"
ADAPTER_DIR = ARTIFACT_DIR / "adapter"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def render_chat(record):
    messages = [
        {
            "role": "system",
            "content": "You are Castalia Mentor, a concise tutor who teaches clearly, uses short examples, and states uncertainty honestly.",
        },
        {"role": "user", "content": record["user"]},
        {"role": "assistant", "content": record["assistant"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"messages": messages, "text": text}

def generate_completion(active_model, active_tokenizer, prompt, max_new_tokens=90):
    messages = [
        {
            "role": "system",
            "content": "You are Castalia Mentor, a concise tutor who teaches clearly, uses short examples, and states uncertainty honestly.",
        },
        {"role": "user", "content": prompt},
    ]
    active_model.eval()
    try:
        FastLanguageModel.for_inference(active_model)
    except Exception:
        pass
    inputs = active_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(active_model.device)
    outputs = active_model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )
    generated_tokens = outputs[0][inputs.shape[-1]:]
    return active_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def keyword_score(text, expected_terms):
    normalized = text.lower()
    hits = sum(term.lower() in normalized for term in expected_terms)
    return round(hits / max(len(expected_terms), 1), 3)

def parameter_report(active_model):
    total_params = 0
    trainable_params = 0
    adapter_rows = []
    for name, param in active_model.named_parameters():
        count = int(param.numel())
        total_params += count
        if param.requires_grad:
            trainable_params += count
        if "lora_" in name:
            adapter_rows.append(
                {
                    "name": name,
                    "parameters": count,
                    "dtype": str(param.dtype),
                    "trainable": bool(param.requires_grad),
                }
            )
    summary_df = pd.DataFrame(
        [
            {"metric": "total_parameters", "value": int(total_params)},
            {"metric": "trainable_parameters", "value": int(trainable_params)},
            {"metric": "trainable_ratio_pct", "value": round(100 * trainable_params / max(total_params, 1), 4)},
            {"metric": "lora_tensor_count", "value": int(len(adapter_rows))},
        ]
    )
    adapter_df = pd.DataFrame(adapter_rows)
    return summary_df, adapter_df

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Build a tiny local conversational dataset

The examples are intentionally small and local so the notebook is runnable. They target one narrow style: concise tutoring explanations with examples and honest scope boundaries.

In [ ]:
raw_examples = [
    {
        "id": "tokens_vs_words",
        "skill": "llm_fundamentals",
        "user": "Explain the difference between tokens and words in two short paragraphs.",
        "assistant": "Words are human-language units, but models usually read text as tokens. A token can be a whole word, part of a word, punctuation, or whitespace pattern depending on the tokenizer.\n\nThat means a 100-word passage is not always 100 tokens. Pricing, context limits, and truncation all depend on token count rather than simple word count.",
    },
    {
        "id": "rag_vs_ft",
        "skill": "model_strategy",
        "user": "When should I use retrieval-augmented generation instead of finetuning?",
        "assistant": "Use RAG when the main problem is missing or changing knowledge. Retrieval is usually faster, cheaper, and easier to update than training.\n\nUse finetuning when you need more stable behavior, style, formatting, or task patterns that prompting and retrieval do not reliably produce.",
    },
    {
        "id": "overfitting_signal",
        "skill": "training_debugging",
        "user": "Give me three warning signs that my finetuning run is overfitting.",
        "assistant": "Three common warning signs are: training loss keeps falling while validation loss stalls or rises, generations start copying phrasing from the training set, and improvements appear on memorized prompts but not on held-out tasks.",
    },
    {
        "id": "adapter_first",
        "skill": "qlora",
        "user": "Why is adapter-first training usually recommended on Colab Pro?",
        "assistant": "Adapters are cheaper because you update a small fraction of weights instead of the full model. That lowers VRAM use, shortens iteration cycles, and makes rollback easier when a run hurts quality.",
    },
    {
        "id": "assistant_only_loss",
        "skill": "datasets",
        "user": "What is assistant-only loss in a chat finetuning dataset?",
        "assistant": "Assistant-only loss means the model is trained to predict the assistant response but not penalized for the user or system turns. This keeps the objective focused on the behavior you want the model to produce.",
    },
    {
        "id": "small_eval_set",
        "skill": "evaluation",
        "user": "Why should I keep a validation split even for a tiny first run?",
        "assistant": "A validation split is your first check against self-deception. Even a small held-out set can reveal whether lower training loss reflects better generalization or just memorization.",
    },
    {
        "id": "qlora_definition",
        "skill": "qlora",
        "user": "Explain QLoRA in plain English.",
        "assistant": "QLoRA keeps the base model in low-bit form so it fits on smaller hardware, then learns small LoRA adapter weights on top. You get most of the iteration benefits of finetuning without updating every parameter.",
    },
    {
        "id": "eval_prompt",
        "skill": "evaluation",
        "user": "What makes a useful held-out prompt for notebook-style evaluation?",
        "assistant": "A useful held-out prompt looks like the task you care about but is not copied from training data. It should be realistic, specific enough to score, and different enough to expose memorization.",
    },
    {
        "id": "learning_rate",
        "skill": "hyperparameters",
        "user": "Why can a too-high learning rate hurt adapter training?",
        "assistant": "A learning rate that is too high can make the adapter updates overshoot useful directions. You often see unstable loss curves, brittle behavior, or regressions on prompts that used to work.",
    },
    {
        "id": "packing_tradeoff",
        "skill": "throughput",
        "user": "What problem does sequence packing solve?",
        "assistant": "Sequence packing reduces wasted padding by combining shorter examples into fuller training sequences. That can improve throughput, but it also makes it easier to hide formatting mistakes if you do not inspect the packed samples.",
    },
    {
        "id": "merge_vs_adapter",
        "skill": "deployment",
        "user": "When would I keep an adapter separate instead of merging it?",
        "assistant": "Keep the adapter separate when you want reversible changes, smaller uploads, or multiple task variants on one base model. Merge when a downstream system needs a single standalone checkpoint.",
    },
    {
        "id": "curriculum",
        "skill": "data_curation",
        "user": "What is curriculum shaping in a finetuning dataset?",
        "assistant": "Curriculum shaping is the deliberate ordering or weighting of examples so the model sees easier or cleaner patterns before harder edge cases. It is a data design choice, not a magic trick.",
    },
    {
        "id": "dropout",
        "skill": "hyperparameters",
        "user": "What does LoRA dropout do?",
        "assistant": "LoRA dropout randomly masks part of the adapter update during training. It can help regularize the run, but on very small datasets a zero-dropout baseline is still common because the adapter is already small.",
    },
    {
        "id": "base_vs_tuned",
        "skill": "evaluation",
        "user": "How should I compare a base model and a tuned model after a first run?",
        "assistant": "Use the same prompts, decoding settings, and scoring rubric for both. Then inspect not only wins but also regressions, verbosity shifts, and cases where the tuned model became less calibrated.",
    },
    {
        "id": "jsonl_reason",
        "skill": "data_curation",
        "user": "Why do many teams store training data as JSONL?",
        "assistant": "JSONL is easy to stream, diff, append, and inspect one record at a time. It also maps cleanly onto dataset tooling for chat, prompt-completion, and preference data.",
    },
    {
        "id": "small_first_run",
        "skill": "training_debugging",
        "user": "Why start with a short first finetuning run instead of a long one?",
        "assistant": "Short runs surface formatting mistakes, unstable settings, and broken eval logic quickly. It is better to debug on a cheap run than discover a pipeline bug after hours of training.",
    },
    {
        "id": "hallucination_scope",
        "skill": "safety",
        "user": "How should a tutor model handle uncertainty when it does not know a detail?",
        "assistant": "It should say what it knows, mark the uncertainty clearly, and avoid inventing specifics. Honest scope boundaries are usually more valuable than confident guessing.",
    },
    {
        "id": "validation_role",
        "skill": "evaluation",
        "user": "Why inspect validation loss and generations together?",
        "assistant": "Loss curves tell you about optimization, but generations tell you about behavior. You need both because a lower loss can still hide worse formatting, weaker calibration, or subtle regressions.",
    },
]

raw_df = pd.DataFrame(raw_examples)
display(raw_df[["id", "skill"]].head())
print("Examples:", len(raw_df))

## Step 3: Render the dataset with the model chat template

The training text should match the model's expected conversation format rather than a hand-written placeholder template.

In [ ]:
dataset = Dataset.from_list(raw_examples)
formatted_dataset = dataset.map(render_chat)

preview_row = formatted_dataset[0]
print(preview_row["text"][:800])

## Step 4: Create a train/validation split

Even for a tiny run we keep a held-out slice. The split is small, but it is enough to make the training loop evaluation-aware.

In [ ]:
splits = formatted_dataset.train_test_split(test_size=0.22, seed=3407)
train_dataset = splits["train"]
eval_dataset = splits["test"]

split_rows = [
    {"split": "train", "rows": len(train_dataset)},
    {"split": "validation", "rows": len(eval_dataset)},
]
display(pd.DataFrame(split_rows))

print("Validation IDs:", [row["id"] for row in eval_dataset])

## Step 5: Define held-out prompts for before/after comparison

These prompts are not exact copies of the training set. They probe the same behavior family with slightly different wording.

In [ ]:
holdout_prompts = [
    {
        "prompt_id": "why_validate",
        "prompt": "Why should a first adapter run still include validation prompts?",
        "expected_terms": ["held-out", "generalization", "memorization"],
    },
    {
        "prompt_id": "adapter_deployment",
        "prompt": "Explain when keeping an adapter separate is better than merging it.",
        "expected_terms": ["reversible", "rollback", "variant"],
    },
    {
        "prompt_id": "uncertainty",
        "prompt": "How should a tutoring model respond when it is unsure about a detail?",
        "expected_terms": ["uncertain", "honest", "invent"],
    },
    {
        "prompt_id": "packing",
        "prompt": "What benefit does sequence packing provide during SFT?",
        "expected_terms": ["padding", "throughput", "shorter"],
    },
]

## Step 5b: Inspect the PEFT adapter footprint before training

 QLoRA keeps the frozen base weights in low-bit form and only updates a small adapter slice. We inspect that slice before training so the parameter budget stays concrete.

In [ ]:
print("PEFT wrapper active:", isinstance(model, PeftModel))

parameter_summary_df, adapter_param_df = parameter_report(model)
display(parameter_summary_df)

adapter_module_df = (
    adapter_param_df.assign(module=adapter_param_df["name"].str.split(".").str[-3])
    .groupby("module", as_index=False)["parameters"]
    .sum()
    .sort_values("parameters", ascending=False)
)

display(adapter_module_df)
display(adapter_param_df.head(12))

## Step 6: Measure the base model before training

We load a clean base copy, generate deterministic completions, and then free that memory before training.

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)
base_tokenizer.padding_side = "right"

base_rows = []
for item in holdout_prompts:
    completion = generate_completion(base_model, base_tokenizer, item["prompt"])
    base_rows.append(
        {
            "prompt_id": item["prompt_id"],
            "prompt": item["prompt"],
            "base_completion": completion,
            "base_anchor_score": keyword_score(completion, item["expected_terms"]),
        }
    )

base_results_df = pd.DataFrame(base_rows)
display(base_results_df[["prompt_id", "base_anchor_score", "base_completion"]])

del base_model
del base_tokenizer
torch.cuda.empty_cache()

## Step 7: Configure the first QLoRA training run

This is intentionally conservative:

- small per-device batch
- gradient accumulation
- short run
- frequent logging
- validation every few steps

In [ ]:
run_dir = ARTIFACT_DIR / "trainer_run"

training_args = TrainingArguments(
    output_dir=str(run_dir),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    evaluation_strategy="steps",
    eval_steps=5,
    save_steps=15,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

print(training_args)

## Step 8: Run the short SFT job

On Colab this should stay small enough for a first end-to-end debug run while still producing visible changes.

In [ ]:
train_result = trainer.train()
trainer.save_state()

log_path = ARTIFACT_DIR / "trainer_log_history.json"
with open(log_path, "w", encoding="utf-8") as handle:
    json.dump(trainer.state.log_history, handle, indent=2)

train_result.metrics

## Step 9: Inspect the training and validation logs

We want to know two things:

1. did optimization run cleanly?
2. did validation stay sensible while train loss dropped?

In [ ]:
log_df = pd.DataFrame(trainer.state.log_history)
display(log_df.tail())

loss_plot_df = log_df[[column for column in ["step", "loss", "eval_loss", "learning_rate"] if column in log_df.columns]].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
loss_plot_df.dropna(subset=["loss"]).plot(x="step", y="loss", ax=axes[0], marker="o", title="Training loss")
if "eval_loss" in loss_plot_df.columns:
    loss_plot_df.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=axes[1], marker="o", color="tab:orange", title="Validation loss")
else:
    axes[1].set_title("Validation loss")
    axes[1].text(0.5, 0.5, "No eval rows logged", ha="center", va="center")
plt.tight_layout()

summary_rows = [
    {"metric": "final_train_loss", "value": round(loss_plot_df["loss"].dropna().iloc[-1], 4)},
]
if "eval_loss" in loss_plot_df.columns and not loss_plot_df["eval_loss"].dropna().empty:
    summary_rows.append({"metric": "final_eval_loss", "value": round(loss_plot_df["eval_loss"].dropna().iloc[-1], 4)})
display(pd.DataFrame(summary_rows))

## Step 10: Compare tuned completions against the baseline

We now generate with the trained adapter using the same prompts and decoding settings as before.

In [ ]:
tuned_rows = []
for item in holdout_prompts:
    completion = generate_completion(model, tokenizer, item["prompt"])
    tuned_rows.append(
        {
            "prompt_id": item["prompt_id"],
            "tuned_completion": completion,
            "tuned_anchor_score": keyword_score(completion, item["expected_terms"]),
        }
    )

tuned_results_df = pd.DataFrame(tuned_rows)
comparison_df = base_results_df.merge(tuned_results_df, on="prompt_id", how="left")
comparison_df["score_delta"] = comparison_df["tuned_anchor_score"] - comparison_df["base_anchor_score"]

display(comparison_df[["prompt_id", "base_anchor_score", "tuned_anchor_score", "score_delta"]])
display(comparison_df[["prompt_id", "prompt", "base_completion", "tuned_completion"]])

## Step 11: Score the held-out behavior shift

This is not a full evaluation suite. It is a lightweight notebook-native check that helps you confirm the adapter is moving in the intended direction.

In [ ]:
aggregate_rows = [
    {
        "model_variant": "base",
        "mean_anchor_score": round(comparison_df["base_anchor_score"].mean(), 3),
    },
    {
        "model_variant": "tuned",
        "mean_anchor_score": round(comparison_df["tuned_anchor_score"].mean(), 3),
    },
]
aggregate_df = pd.DataFrame(aggregate_rows)
display(aggregate_df)

ax = aggregate_df.plot(
    x="model_variant",
    y="mean_anchor_score",
    kind="bar",
    legend=False,
    figsize=(6, 4),
    color=["tab:gray", "tab:green"],
    title="Hold-out anchor score",
)
ax.set_ylabel("score")
plt.xticks(rotation=0)
plt.tight_layout()

## Step 12: Save the adapter artifacts

We keep the adapter separate because that is the default practical workflow: cheaper to store, easier to version, and easy to compare against the untouched base model.

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

saved_files = sorted(path.name for path in ADAPTER_DIR.iterdir())
display(pd.DataFrame({"saved_file": saved_files}))

## Step 12b: Inspect the saved adapter metadata

 A saved adapter is not just weight tensors. The PEFT config records rank, target modules, and loading details for later reuse.

In [ ]:
adapter_config_path = ADAPTER_DIR / "adapter_config.json"
adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
adapter_config_df = pd.DataFrame(
    [{"field": key, "value": str(value)} for key, value in adapter_config.items()]
).sort_values("field")
display(adapter_config_df.head(20))

## Step 13: Write a short run summary

A good first run leaves a paper trail. Save what changed, what improved, and what to adjust next.

In [ ]:
run_summary = {
    "train_rows": len(train_dataset),
    "validation_rows": len(eval_dataset),
    "max_steps": training_args.max_steps,
    "learning_rate": training_args.learning_rate,
    "mean_base_anchor_score": float(round(comparison_df["base_anchor_score"].mean(), 3)),
    "mean_tuned_anchor_score": float(round(comparison_df["tuned_anchor_score"].mean(), 3)),
    "trainable_parameters": int(parameter_summary_df.loc[parameter_summary_df["metric"] == "trainable_parameters", "value"].iloc[0]),
    "trainable_ratio_pct": float(parameter_summary_df.loc[parameter_summary_df["metric"] == "trainable_ratio_pct", "value"].iloc[0]),
    "adapter_dir": str(ADAPTER_DIR),
}

summary_path = ARTIFACT_DIR / "run_summary.json"
with open(summary_path, "w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2)

run_summary

## Key takeaways

- **QLoRA is the practical default** for a first Colab run.
- **Small local data is enough** to validate the full pipeline.
- **Training logs plus held-out prompts** are better than loss alone.
- **Adapters stay separate by default** until you have a reason to merge.